# Fix: re-extract TCGA CONCH v1.5 five-crop features with FiveCrop(256)

**Why.** `dataset/Features/Conch_v1_5_fivecrop_Updated` is not one consistent feature set. A scan on 2026-10-11 recomputed 2 patches per slide (all 5 crops) for all 417 slides. Every slide matched exactly one crop size at cosine 1.0000:

| crop size | slides | MSI-H | patches | written |
|---|---|---|---|---|
| 224 | 182 (TCGA-3L-AA1B … TCGA-AY-A71X) | 26 | 527,391 | 2025-07-31 → 2025-08-01 11:41 |
| 256 | 235 (TCGA-AY-A8YK onward) | 34 | 850,542 | 2025-08-01 11:49 → 2025-08-06 |

The run was restarted on 2025-08-01 with `FiveCrop(224)` changed to `FiveCrop(256)`, and the finished slides were kept. Every other encoder and cohort (including PAIP CONCH v1.5) uses 256.

**What this notebook does.** It re-extracts the 182 slides with `FiveCrop(256)` into a **new** folder, `Conch_v1_5_fivecrop_256`. It then hard-links the 235 correct slides into the same folder, so the new folder is one complete, consistent set. The old folder is not touched. Hard links take no space and survive deleting the old folder.

**Same settings as the original run.** Everything except the crop size is copied from `Feature_Extraction_Conch_fivecrop.ipynb`:
- model: TITAN `return_conch()` (CONCH v1.5);
- transform: TITAN's (Resize 448 bilinear, CenterCrop 448, ImageNet mean/std);
- crops: `FiveCrop(256)`;
- precision: fp32 `torch.no_grad()`, PyTorch defaults (no TF32);
- output: one float32 `(5, 768)` tensor per patch, with the same file names.

Check A proves this by reproducing the stored 256-slide files.

**Speed.** fp32 CONCH v1.5 at 448 px is GPU-bound at about 58 crops/s, which is 11.7 patches/s on the RTX 4090. That rate is the same from 4 to 32 patches per batch, and at 64 it collapses. The only faster options, fp16 or TF32, would change the features, so 16 patches (80 crops) per batch is used. The 527,391 patches take **about 12.5 h**. The run is resumable.

**Run order.**
1. Restart the kernel and run sections 1–3.
2. Run the checks (section 4). Go on only if every line says `PASS`.
3. Run the extraction (section 5).
4. Run the link and verification cells (section 6).
5. Delete the old folder only after the final verification prints `ALL PASS`.

## 1. Settings

In [ ]:
import os
# Gated models load from the local Hugging Face cache. Must be set before transformers / torchvision import huggingface_hub.
os.environ["HF_HUB_OFFLINE"] = "1"

import random
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

TCGA = Path(r"D:\Aamir Gulzar\KSA_project2\dataset")
PATCH_ROOT = TCGA / "patch_data"
NON_WHITE_CSV = TCGA / "final_merged_without_white.csv"
OLD = TCGA / "Features" / "Conch_v1_5_fivecrop_Updated"   # read only
NEW = TCGA / "Features" / "Conch_v1_5_fivecrop_256"       # written by this notebook

LAST_224_SLIDE = "TCGA-AY-A71X_nonMSIH"   # slides sorted up to and including this one were made with FiveCrop(224)
CROP = 256
BATCH_PATCHES = 16    # 80 crops per GPU call: fastest fp32 setting measured (58 crops/s, ~4 GB)
NUM_WORKERS = 8       # PNG read + transform threads (the GPU needs only ~12 patches/s)
PASS_COSINE = 0.9999

DEVICE = torch.device("cuda")
print("TF32 matmul (must be False):", torch.backends.cuda.matmul.allow_tf32)

nw = pd.read_csv(NON_WHITE_CSV, usecols=["patch_location", "patch_name"])
nw["slide"] = nw["patch_location"].str.replace("\\", "/", regex=False).str.split("/").str[-2]
PATCHES = {s: sorted(g) for s, g in nw.groupby("slide")["patch_name"]}   # slide -> non-white PNG names
SLIDES = sorted(PATCHES)
SLIDES_224 = [s for s in SLIDES if s <= LAST_224_SLIDE]
SLIDES_256 = [s for s in SLIDES if s > LAST_224_SLIDE]
print(f"{len(SLIDES)} slides: {len(SLIDES_224)} to re-extract ({sum(len(PATCHES[s]) for s in SLIDES_224):,} patches), "
      f"{len(SLIDES_256)} already 256")

## 2. Model (copied from `Feature_Extraction_Conch_fivecrop.ipynb`)

In [ ]:
from transformers import AutoModel

_titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
conch, eval_transform = _titan.return_conch()
del _titan
conch = conch.to(DEVICE).eval()
print(eval_transform)


def fivecrop(img, size=CROP):
    """FiveCrop(size) -> eval_transform each crop -> (5, 3, 448, 448). Same as the original notebook."""
    return torch.stack([eval_transform(c) for c in transforms.FiveCrop(size)(img)])


def embed(crops):
    """(N, 3, 448, 448) -> (N, 768) float32 on CPU. fp32, no_grad: same as the original notebook."""
    with torch.no_grad():
        return conch(crops.to(DEVICE)).cpu()


def open_patch(slide, png):
    return Image.open(PATCH_ROOT / slide / png).convert("RGB")

## 3. Checks

| check | what it proves | pass if |
|---|---|---|
| **A** | This code reproduces stored files of slides made with 256 crops. Model, transform and precision are identical. | every crop cosine ≥ 0.9999 |
| **B** | Re-confirms, for all 417 slides (1 patch each), which crop size made the stored file. The 182/235 split must come out again. | split reproduced |
| **C** | Batching 16 patches (80 crops) gives the same features as one patch (5 crops) per call, as the original run did. | cosine ≥ 0.9999 |

In [ ]:
def cos_min(a, b):
    return F.cosine_similarity(a.float(), b.float(), dim=1).min().item()


def check_A(n_slides=4, n_patches=8):
    ok = True
    for slide in random.Random(0).sample(SLIDES_256, n_slides):
        worst, diff = 1.0, 0.0
        for png in random.Random(1).sample(PATCHES[slide], n_patches):
            stored = torch.load(OLD / slide / (png[:-4] + ".pt"), map_location="cpu")
            new = embed(fivecrop(open_patch(slide, png)))
            worst, diff = min(worst, cos_min(new, stored)), max(diff, (new - stored).abs().max().item())
        ok &= worst >= PASS_COSINE
        print(f"  A {slide}: min cosine {worst:.7f}, max abs diff {diff:.2e}  {'PASS' if worst >= PASS_COSINE else 'FAIL'}")
    return ok


def check_B():
    found = {}
    for slide in tqdm(SLIDES, desc="Check B"):
        png = random.Random(slide).choice(PATCHES[slide])
        stored = torch.load(OLD / slide / (png[:-4] + ".pt"), map_location="cpu")
        img = open_patch(slide, png)
        c256, c224 = cos_min(embed(fivecrop(img, 256)), stored), cos_min(embed(fivecrop(img, 224)), stored)
        found[slide] = 256 if c256 >= PASS_COSINE else 224 if c224 >= PASS_COSINE else None
    ok = ([s for s in SLIDES if found[s] == 224] == SLIDES_224 and [s for s in SLIDES if found[s] == 256] == SLIDES_256)
    unknown = [s for s in SLIDES if found[s] is None]
    print(f"  B 224: {sum(v == 224 for v in found.values())}, 256: {sum(v == 256 for v in found.values())}, "
          f"neither: {len(unknown)} {unknown[:5]}  {'PASS' if ok else 'FAIL'}")
    return ok


def check_C(n_patches=BATCH_PATCHES):
    slide = SLIDES_224[0]
    pngs = random.Random(2).sample(PATCHES[slide], n_patches)
    x = torch.cat([fivecrop(open_patch(slide, p)) for p in pngs])          # (16*5, 3, 448, 448)
    batched = embed(x)
    single = torch.cat([embed(x[i:i + 5]) for i in range(0, len(x), 5)])   # 5 crops per call, as originally
    c = cos_min(batched, single)
    print(f"  C batch of {n_patches} patches vs one patch per call: min cosine {c:.7f}  {'PASS' if c >= PASS_COSINE else 'FAIL'}")
    return c >= PASS_COSINE

In [ ]:
results = {"A": check_A(), "B": check_B(), "C": check_C()}
print("\nALL CHECKS PASS - safe to run section 4" if all(results.values()) else f"\nSTOP - failed: {[k for k, v in results.items() if not v]}")

## 4. Re-extract the 182 slides
- Reader threads open each PNG once and build its five crops ahead of the GPU.
- The GPU embeds 16 patches (80 crops) per call.
- Each patch's `(5, 768)` is saved through a `.tmp` file and a rename, so an interrupted run never leaves a half-written file.
- Rerunning the cell continues where it stopped.

In [ ]:
def save_atomic(tensor, path):
    tmp = str(path) + ".tmp"
    torch.save(tensor, tmp)
    os.replace(tmp, path)


def reextract(slides=None, out_root=NEW):
    slides = SLIDES_224 if slides is None else slides
    jobs = []
    for slide in slides:
        d = out_root / slide
        d.mkdir(parents=True, exist_ok=True)
        for f in d.glob("*.tmp"):
            f.unlink()
        done = {f.name[:-3] for f in os.scandir(d) if f.name.endswith(".pt")}
        jobs += [(slide, png) for png in PATCHES[slide] if png[:-4] not in done]
    total = sum(len(PATCHES[s]) for s in slides)
    print(f"{len(slides)} slides, {total:,} patches: {total - len(jobs):,} done, {len(jobs):,} to do -> {out_root}")
    if not jobs:
        return 0

    failed = 0

    def collect(fut):
        try:
            fut.result()
            return 0
        except Exception as e:
            tqdm.write(f"  [save failed] {type(e).__name__}: {e}")
            return 1

    with ThreadPoolExecutor(NUM_WORKERS) as readers, ThreadPoolExecutor(2) as savers:
        pending, saving = deque(), deque()
        it = iter(jobs)

        def submit():
            j = next(it, None)
            if j is not None:
                pending.append((j, readers.submit(lambda jj: fivecrop(open_patch(*jj)), j)))

        for _ in range(BATCH_PATCHES * 4):
            submit()
        pbar = tqdm(total=len(jobs), desc="Re-extracting", unit="patch")
        while pending:
            batch = []
            while pending and len(batch) < BATCH_PATCHES:
                job, fut = pending.popleft()
                submit()
                try:
                    batch.append((job, fut.result()))
                except Exception as e:
                    failed += 1
                    pbar.update(1)
                    tqdm.write(f"  [read failed] {job}: {e}")
            if not batch:
                continue
            try:
                feats = embed(torch.cat([x for _, x in batch]))       # (n*5, 768)
            except Exception as e:                                    # log and go on; rerun retries these
                failed += len(batch)
                pbar.update(len(batch))
                tqdm.write(f"  [GPU failed] {len(batch)} patches from {batch[0][0][0]}: {type(e).__name__}: {e}")
                torch.cuda.empty_cache()
                continue
            for k, ((slide, png), _) in enumerate(batch):
                path = out_root / slide / (png[:-4] + ".pt")
                saving.append(savers.submit(save_atomic, feats[5 * k:5 * k + 5].clone(), path))   # (5, 768)
            while saving and (saving[0].done() or len(saving) > 2000):
                failed += collect(saving.popleft())
            pbar.update(len(batch))
        for fut in saving:
            failed += collect(fut)
        pbar.close()
    print(f"done: {len(jobs) - failed:,} saved, {failed} failed" + ("  - run the cell again to retry" if failed else ""))
    return failed

In [ ]:
reextract()

## 5. Complete the new folder and verify it
`link_good_slides()` hard-links every file of the 235 correct slides from the old folder into the new one. No data is copied, and no extra disk space is used. After that, the final check must print `ALL PASS`:
- all 417 slides, with file names exactly matching the non-white CSV;
- every file a finite float32 `(5, 768)`;
- re-extracted slides: a sample, recomputed one patch at a time, matches;
- re-extracted slides: none is identical to the old 224 file;
- linked slides: identical to the old files.

In [ ]:
def link_good_slides(out_root=NEW, slides=None):
    n_new = 0
    for slide in tqdm(SLIDES_256 if slides is None else slides, desc="Linking 256 slides"):
        d = out_root / slide
        d.mkdir(parents=True, exist_ok=True)
        for png in PATCHES[slide]:
            dst = d / (png[:-4] + ".pt")
            if not dst.exists():
                os.link(OLD / slide / (png[:-4] + ".pt"), dst)
                n_new += 1
    print(f"linked {n_new:,} new files")


link_good_slides()

In [ ]:
def verify(out_root=NEW, slides=None, n_recompute_slides=20):
    slides = SLIDES if slides is None else slides
    ok = True
    bad_names, bad_tensors = [], 0
    for slide in tqdm(slides, desc="Verifying files"):
        d = out_root / slide
        got = {f.name for f in os.scandir(d)} if d.is_dir() else set()
        want = {p[:-4] + ".pt" for p in PATCHES[slide]}
        if got != want:
            bad_names.append(slide)
            continue
        for name in random.Random(slide).sample(sorted(want), min(5, len(want))):   # sampled tensor check
            t = torch.load(d / name, map_location="cpu")
            bad_tensors += not (t.shape == (5, 768) and t.dtype == torch.float32 and torch.isfinite(t).all())
    print(f"  slides with wrong file set: {len(bad_names)} {bad_names[:5]}")
    print(f"  bad tensors in sample: {bad_tensors}")
    ok &= not bad_names and bad_tensors == 0

    redone = [s for s in slides if s in SLIDES_224]
    worst, same_as_old = 1.0, 0
    for slide in random.Random(3).sample(redone, min(n_recompute_slides, len(redone))):
        png = random.Random(4).choice(PATCHES[slide])
        new = torch.load(out_root / slide / (png[:-4] + ".pt"), map_location="cpu")
        worst = min(worst, cos_min(new, embed(fivecrop(open_patch(slide, png)))))
        same_as_old += torch.equal(new, torch.load(OLD / slide / (png[:-4] + ".pt"), map_location="cpu"))
    if redone:
        print(f"  re-extracted slides: recomputed min cosine {worst:.7f}; identical to old 224 file: {same_as_old}")
        ok &= worst >= PASS_COSINE and same_as_old == 0

    linked = [s for s in slides if s in SLIDES_256]
    differ = 0
    for slide in random.Random(5).sample(linked, min(20, len(linked))):
        png = random.Random(6).choice(PATCHES[slide])
        differ += not torch.equal(torch.load(out_root / slide / (png[:-4] + ".pt")), torch.load(OLD / slide / (png[:-4] + ".pt")))
    if linked:
        print(f"  linked slides differing from old file: {differ}")
        ok &= differ == 0
    print("\nALL PASS - the old folder can be deleted" if ok else "\nFAIL - do not delete the old folder")
    return ok


verify()